In [0]:
# Create a balanced dataset
# Each customer has equal number of transactions

from pyspark.sql.functions import col, count

balanced_data = []
customers = ["Alice", "Bob", "Charlie", "Diana"]

# Each customer gets 250 transactions = 1000 total
for customer in customers:
    for i in range(250):
        balanced_data.append((
            customer,
            float(i * 100),
            "credit"
        ))

balanced_df = spark.createDataFrame(
    balanced_data,
    ["customer", "amount", "type"]
)

print(f"Total records: {balanced_df.count()}")
#print(f"Total partitions: {balanced_df.rdd.getNumPartitions()}")

# Check distribution per customer
balanced_df.groupBy("customer").count().show()

In [0]:
# Create a SKEWED dataset
# One customer has WAY more data than others

skewed_data = []

# Alice gets 9700 transactions (skewed!)
for i in range(9700):
    skewed_data.append((
        "Alice",
        float(i * 100),
        "credit"
    ))

# Others get only 100 each
for customer in ["Bob", "Charlie", "Diana"]:
    for i in range(100):
        skewed_data.append((
            customer,
            float(i * 100),
            "credit"
        ))

skewed_df = spark.createDataFrame(
    skewed_data,
    ["customer", "amount", "type"]
)

print(f"Total records: {skewed_df.count()}")

# Check distribution — see the imbalance!
skewed_df.groupBy("customer").count().orderBy("count", ascending=False).show()

In [0]:
import time

# Test 1 — Balanced data
start = time.time()
balanced_df.groupBy("customer").sum("amount").show()
balanced_time = time.time() - start
print(f"Balanced time: {balanced_time:.2f} seconds")

# Test 2 — Skewed data
start = time.time()
skewed_df.groupBy("customer").sum("amount").show()
skewed_time = time.time() - start
print(f"Skewed time: {skewed_time:.2f} seconds")

print(f"\nSkewed took {skewed_time/balanced_time:.1f}x longer!")

In [0]:
# Broadcast Join — for small tables
from pyspark.sql.functions import broadcast

# Small lookup table (customer details)
customer_details = spark.createDataFrame([
    ("Alice",   "Premium"),
    ("Bob",     "Standard"),
    ("Charlie", "Standard"),
    ("Diana",   "Premium")
], ["customer", "tier"])

print(f"Small table size: {customer_details.count()} rows")

# Normal join vs Broadcast join
# Broadcast = send small table to ALL workers
# No shuffling needed! ✅

result = skewed_df.join(
    broadcast(customer_details),  # ← broadcast hint!
    "customer"
)

result.groupBy("customer", "tier").count().show()

In [0]:
# Check current shuffle partitions setting
print(spark.conf.get("spark.sql.shuffle.partitions"))